# 05 - Distribution Shift and KD Failure Modes

The first four notebooks showed distillation working under the **same distribution**: teacher
and student see the same kinds of inputs at training and test time. This notebook breaks that
assumption.

**The failure mode.** When a distilled student is deployed on inputs from groups that were
rare or absent during training, the student fails *more severely* than the teacher -- even
when both were trained on the same data. The mechanism: a smaller student has less capacity,
so it latches onto **spurious correlations** (cheap superficial patterns that happen to
predict the label at training time) rather than robust causal features. Distribution shift
breaks the spurious correlation; the student falls apart while the teacher degrades more
gracefully because it also learned genuine shape features.

**Evidence.** arXiv 2506.02294 (June 2025) documents this on real benchmarks: teacher
worst-group accuracy 94.6 % vs. student 66.1 % on CelebA. The *causal* feature (gender)
and the *spurious* feature (hair colour) are decoupled at test time; the student chases the
spurious one.

**What we build here.** A Color-MNIST experiment that reproduces the mechanism cleanly:

1. Colour each digit's background **red** (even class) or **blue** (odd class) with 90 %
   probability at training time -- the spurious correlation.
2. Train a **teacher CNN** on this biased data; distill a smaller **student CNN** from it.
3. Evaluate on two test groups: **Group A** (colour matches training pattern) and **Group B**
   (colours reversed -- the distribution shift).
4. Measure **per-group teacher-student agreement** -- the production monitoring metric that
   drops measurably for shift-affected inputs *without needing ground-truth labels*.

In [ ]:
import time
from pathlib import Path

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
from torchvision import datasets, transforms
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(0)
np.random.seed(0)
DEVICE = torch.device('cpu')

# Resolve repo data/ regardless of where the notebook runs from.
_r = Path.cwd()
while not (_r / 'pyproject.toml').exists() and _r != _r.parent:
    _r = _r.parent
DATA_ROOT = str(_r / 'data')
print('data root:', DATA_ROOT)
print('torch threads:', torch.get_num_threads())

## Constructing Color-MNIST

Standard MNIST digits have a black background. We replace that background with a solid
colour -- **red** or **blue** -- to create a spurious signal.

- **Causal feature**: digit shape (which of the 10 classes is drawn).
- **Spurious feature**: background colour (red vs. blue).

**Training (biased, 90 % correlation)**  
Even-class digits (0,2,4,6,8): 90 % red background.  
Odd-class digits (1,3,5,7,9): 90 % blue background.  
The remaining 10 % get the 'wrong' colour -- just enough noise to prevent perfect correlation.

**Group A (correlated test)**: even -> red, odd -> blue. Matches the dominant training pattern.

**Group B (anti-correlated test)**: even -> blue, odd -> red. Colour now *opposes* the label.

A model that learned 'red means even' does well on Group A and fails on Group B. A model
that learned genuine digit shape degrades only modestly on both.

In [ ]:
def colorize(img_gray, color):
    """img_gray: (1, H, W) float [0,1]. Returns (3, H, W) with coloured background."""
    rgb = img_gray.repeat(3, 1, 1).clone()
    bg = img_gray[0] < 0.15   # MNIST background pixels are near 0
    if color == 'red':
        rgb[0][bg] = 0.85
        rgb[1][bg] = 0.10
        rgb[2][bg] = 0.10
    else:                     # blue
        rgb[0][bg] = 0.10
        rgb[1][bg] = 0.10
        rgb[2][bg] = 0.85
    return rgb


def assign_color(label, group, rng, corr_prob):
    corr = 'red'  if label % 2 == 0 else 'blue'
    anti = 'blue' if label % 2 == 0 else 'red'
    if group == 'A':  return corr
    if group == 'B':  return anti
    return corr if rng.random() < corr_prob else anti   # group='train': stochastic


def build_color_mnist(raw_ds, n_per_class, group, rng=None, corr_prob=0.90):
    if rng is None:
        rng = np.random.default_rng(0)
    labels_all = raw_ds.targets.numpy()
    xs, ys = [], []
    for c in range(10):
        idxs = np.where(labels_all == c)[0][:n_per_class]
        for idx in idxs:
            img, lbl = raw_ds[int(idx)]
            xs.append(colorize(img, assign_color(lbl, group, rng, corr_prob)))
            ys.append(lbl)
    return torch.stack(xs), torch.tensor(ys)


raw_tfm  = transforms.ToTensor()   # keeps [0,1]; we handle channels manually
raw_train = datasets.MNIST(DATA_ROOT, train=True,  download=True, transform=raw_tfm)
raw_test  = datasets.MNIST(DATA_ROOT, train=False, download=True, transform=raw_tfm)

N_TRAIN = 200   # per class -> 2000 training images total
N_TEST  = 80    # per class per group -> 800 images per group

train_rng = np.random.default_rng(0)
Xtr,   Ytr   = build_color_mnist(raw_train, N_TRAIN, group='train', rng=train_rng)
Xte_A, Yte_A = build_color_mnist(raw_test,  N_TEST,  group='A')
Xte_B, Yte_B = build_color_mnist(raw_test,  N_TEST,  group='B')

print('train:', tuple(Xtr.shape))
print('Group A (correlated):', tuple(Xte_A.shape))
print('Group B (anti-correlated):', tuple(Xte_B.shape))
# Sanity: red channel should be high for even-class training images (90% are red)
red_avg = Xtr[Ytr % 2 == 0, 0].mean().item()
print(f'Training even-class red channel avg: {red_avg:.3f}  (should be ~0.4+ for red-biased)')

In [ ]:
# Visualise one example from each (group, parity) combination so the colour signal is visible.
combos = [
    (Xte_A, Yte_A, 0, 'Group A\neven (red bg)'),
    (Xte_A, Yte_A, 1, 'Group A\nodd (blue bg)'),
    (Xte_B, Yte_B, 0, 'Group B\neven (blue bg) -- shift'),
    (Xte_B, Yte_B, 1, 'Group B\nodd (red bg) -- shift'),
]
fig, axes = plt.subplots(1, 4, figsize=(9, 2.5))
for ax, (X, Y, parity, title) in zip(axes, combos):
    mask = (Y % 2 == parity)
    img = X[mask][0].permute(1, 2, 0).numpy()
    ax.imshow(img, vmin=0, vmax=1)
    ax.set_title(title, fontsize=8)
    ax.axis('off')
fig.suptitle('Background colour is the spurious feature  |  digit shape is the causal feature',
             fontsize=8, y=1.02)
plt.tight_layout()
plt.show()

## Model architectures

Both models take **3-channel** (RGB) images as input because colour is now part of the signal.

The **teacher** is a standard two-conv CNN with two fully-connected layers -- the same style
used in notebooks 02 and 04, but with 3-channel input. It has enough capacity to learn both
colour cues and shape features.

The **student** is a smaller CNN: fewer filters per layer, no second FC layer. With less
capacity it is pushed toward the *easiest* available signal -- and background colour is far
easier to learn from a few thousand images than the subtle stroke patterns that distinguish
a 3 from an 8. That is the capacity asymmetry the spurious-correlation failure exploits.

In [ ]:
class TeacherCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 16, 3, padding=1)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc1   = nn.Linear(32 * 7 * 7, 64)
        self.fc2   = nn.Linear(64, 10)

    def forward(self, x):
        h = self.pool(F.relu(self.conv1(x)))
        h = self.pool(F.relu(self.conv2(h)))
        h = h.flatten(1)
        h = F.relu(self.fc1(h))
        return self.fc2(h)


class StudentCNN(nn.Module):
    """Smaller CNN: fewer filters, no second FC -- pushed toward cheap colour features."""
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 8, 3, padding=1)
        self.conv2 = nn.Conv2d(8, 16, 3, padding=1)
        self.pool  = nn.MaxPool2d(2)
        self.fc    = nn.Linear(16 * 7 * 7, 10)

    def forward(self, x):
        h = self.pool(F.relu(self.conv1(x)))
        h = self.pool(F.relu(self.conv2(h)))
        h = h.flatten(1)
        return self.fc(h)


def count_params(m):
    return sum(p.numel() for p in m.parameters())

print('teacher params:', count_params(TeacherCNN()))
print('student params:', count_params(StudentCNN()))
print('ratio: %.1fx' % (count_params(TeacherCNN()) / count_params(StudentCNN())))

In [ ]:
BATCH = 128
train_ds = TensorDataset(Xtr, Ytr)
train_loader = DataLoader(train_ds, batch_size=BATCH, shuffle=True)
# Ordered loader for caching teacher logits -- must match Xtr/Ytr index order.
train_ordered = DataLoader(train_ds, batch_size=256, shuffle=False)
loader_A = DataLoader(TensorDataset(Xte_A, Yte_A), batch_size=256, shuffle=False)
loader_B = DataLoader(TensorDataset(Xte_B, Yte_B), batch_size=256, shuffle=False)


@torch.no_grad()
def accuracy(model, loader):
    model.eval()
    correct = total = 0
    for x, y in loader:
        correct += (model(x).argmax(1) == y).sum().item()
        total   += y.size(0)
    return correct / total


def train_ce(model, epochs, lr=1e-3):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    model.train()
    for ep in range(epochs):
        total_loss = total_n = 0
        for x, y in train_loader:
            loss = F.cross_entropy(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * x.size(0)
            total_n    += x.size(0)
        if (ep + 1) % 3 == 0 or ep == epochs - 1:
            print(f'  epoch {ep+1}/{epochs}  loss={total_loss/total_n:.4f}')
    return model

## Step 1 -- Train the teacher on biased data

The teacher trains with ordinary cross-entropy on the 2000-image biased training set. It
has enough capacity to learn *both* the colour shortcut and some genuine shape features.
On Group A it will do well because colour aligns with labels. On Group B colour opposes
labels, so any reliance on colour hurts -- but the teacher's shape knowledge partially
compensates.

In [ ]:
T_EPOCHS = 12
t0 = time.time()
torch.manual_seed(0)
teacher = TeacherCNN()
train_ce(teacher, T_EPOCHS)
acc_t_A = accuracy(teacher, loader_A)
acc_t_B = accuracy(teacher, loader_B)
print(f'Teacher  Group A (correlated):      {acc_t_A:.4f}')
print(f'Teacher  Group B (anti-correlated): {acc_t_B:.4f}')
print(f'Gap A-B: {acc_t_A - acc_t_B:+.4f}  (teacher pays some cost for spurious colour reliance)')
print(f'elapsed: {time.time()-t0:.1f}s')

## Step 2 -- Distill the student

We cache the teacher's logits over the training set (one pass, then frozen), then train
the student with the standard Hinton objective: a blend of soft-target KL divergence
and hard-label cross-entropy.

Because the teacher was trained on biased data, its soft targets *encode the spurious
correlation*. The student, smaller and less able to learn robust shape features, inherits
those biased targets and leans on the easy colour signal even more heavily. The expected
result: the student's Group A accuracy is reasonable, but its Group B accuracy drops
further than the teacher's -- a wider gap despite identical training data.

In [ ]:
# Cache teacher logits over the training set (ordered -- matches Xtr/Ytr indices).
teacher.eval()
cached = []
with torch.no_grad():
    for x, _ in train_ordered:
        cached.append(teacher(x))
teacher_logits_all = torch.cat(cached, 0)   # (N_TRAIN*10, 10)
print('cached teacher logits:', tuple(teacher_logits_all.shape))

T_KD   = 4.0
ALPHA  = 0.7
N      = Xtr.size(0)
S_EPOCHS = 25


def train_distilled(epochs=S_EPOCHS, T=T_KD, alpha=ALPHA):
    torch.manual_seed(0)
    student = StudentCNN()
    opt = torch.optim.Adam(student.parameters(), lr=1e-3)
    student.train()
    for ep in range(epochs):
        perm = torch.randperm(N)
        total_loss = total_n = 0
        for i in range(0, N, BATCH):
            idx = perm[i:i + BATCH]
            s_logits = student(Xtr[idx])
            t_logits = teacher_logits_all[idx]
            soft = F.kl_div(
                F.log_softmax(s_logits / T, dim=1),
                F.softmax(t_logits / T, dim=1),
                reduction='batchmean',
            ) * (T * T)
            hard = F.cross_entropy(s_logits, Ytr[idx])
            loss = alpha * soft + (1.0 - alpha) * hard
            opt.zero_grad(); loss.backward(); opt.step()
            total_loss += loss.item() * idx.size(0)
            total_n    += idx.size(0)
        if (ep + 1) % 5 == 0 or ep == epochs - 1:
            print(f'  epoch {ep+1}/{epochs}  loss={total_loss/total_n:.4f}')
    return student


t0 = time.time()
student = train_distilled()
acc_s_A = accuracy(student, loader_A)
acc_s_B = accuracy(student, loader_B)
print(f'Student (distilled)  Group A:      {acc_s_A:.4f}')
print(f'Student (distilled)  Group B:      {acc_s_B:.4f}')
print(f'Gap A-B: {acc_s_A - acc_s_B:+.4f}  (student gap should be wider than teacher gap)')
print(f'elapsed: {time.time()-t0:.1f}s')

## Step 3 -- Per-group agreement: the detection signal

In production you often do *not* have ground-truth labels for incoming data -- that is the
whole point of the model. But you still have both the teacher and the deployed student.

**Agreement rate**: fraction of examples where teacher and student predict the *same* class.
This requires no labels. If agreement is high on one data slice and low on another, the
low-agreement slice is likely under distribution shift.

Expected pattern:
- **Group A** (correlated, seen at training): high accuracy for both models, high agreement.
- **Group B** (anti-correlated, shift): both models degrade, but the student degrades more,
  so agreement *also* drops -- the teacher and student start disagreeing.

The agreement drop on Group B is the early-warning signal. It can be monitored continuously
on live traffic without labels, sliced by any available metadata (geography, device type,
time of day), and a sustained drop flags which slice to label and retrain on first.

In [ ]:
@torch.no_grad()
def agreement_rate(model_a, model_b, loader):
    """Fraction of examples where both models predict the same class. No labels needed."""
    model_a.eval(); model_b.eval()
    agree = total = 0
    for x, _ in loader:
        agree += (model_a(x).argmax(1) == model_b(x).argmax(1)).sum().item()
        total += x.size(0)
    return agree / total


agr_A = agreement_rate(teacher, student, loader_A)
agr_B = agreement_rate(teacher, student, loader_B)

print('Per-group summary')
print(f'  {"":20s}  {"Group A":>10s}  {"Group B":>10s}  {"Gap (B-A)":>10s}')
print(f'  {"teacher accuracy":20s}  {acc_t_A:>10.4f}  {acc_t_B:>10.4f}  {acc_t_B-acc_t_A:>+10.4f}')
print(f'  {"student accuracy":20s}  {acc_s_A:>10.4f}  {acc_s_B:>10.4f}  {acc_s_B-acc_s_A:>+10.4f}')
print(f'  {"t-s agreement":20s}  {agr_A:>10.4f}  {agr_B:>10.4f}  {agr_B-agr_A:>+10.4f}')

# Chart: three metrics, two groups.
groups   = ['Group A\n(correlated)', 'Group B\n(shift)']
metrics  = {
    'teacher accuracy':   ([acc_t_A, acc_t_B], '#4c72b0'),
    'student accuracy':   ([acc_s_A, acc_s_B], '#c44e52'),
    'teacher-student\nagreement': ([agr_A, agr_B], '#8c8c8c'),
}

x = np.arange(len(groups))
width = 0.22
offsets = [-1, 0, 1]

fig, ax = plt.subplots(figsize=(7, 4.5))
for (label, (vals, color)), offset in zip(metrics.items(), offsets):
    bars = ax.bar(x + offset * width, vals, width, label=label, color=color, alpha=0.85)
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2, val + 0.012,
                f'{val:.3f}', ha='center', va='bottom', fontsize=8)

ax.set_xticks(x)
ax.set_xticklabels(groups)
ax.set_ylim(0, 1.15)
ax.set_ylabel('rate')
ax.set_title('Distribution shift: accuracy and teacher-student agreement per group')
ax.legend(loc='upper right', fontsize=8)
ax.grid(True, axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print(f'\nAgreement drop (A->B): {agr_B - agr_A:+.4f}')
print('This is the unlabelled detection signal: no ground-truth labels required.')

## Takeaways

1. **KD does not protect against distribution shift -- it can amplify it.** The teacher
   already encodes the spurious correlation in its soft targets. The student, with less
   capacity, relies on that shortcut more heavily and falls further on shifted data.
   Tighter student-teacher alignment during distillation transfers the teacher's biases
   alongside the teacher's knowledge.

2. **Per-group accuracy drop is the symptom; per-group agreement drop is the detectable
   signal.** In production you cannot always measure accuracy (you need labels for that),
   but you can measure teacher-student agreement on live traffic. A sustained agreement drop
   on a data slice -- without any labels -- is an early indicator that the student is
   behaving differently from the teacher on that slice.

3. **Monitoring prescription.** Slice live predictions by any available metadata (device,
   geography, time window, user cohort). For each slice compute the rolling
   teacher-student agreement rate. Alert when agreement on a slice drops more than a
   threshold below the baseline measured at deployment time. Then label a sample of that
   slice and prioritise it for retraining.

4. **Recovery options are limited and modest.** The surviving literature (arXiv 2309.11446,
   ICCV 2023 OOD-CV Workshop) shows weight-averaging over a single training trajectory
   (WAKD) recovers +0.8--1.6 pp across held-out domains. Jacobian matching (Wu et al.,
   ICML 2024) enables partial mechanism transfer but does not close the gap. No verified
   method eliminates the fidelity-generalisation trade-off under distribution shift.

5. **The fidelity paradox still applies.** Notebook 02 showed that tighter distribution
   matching can *hurt* IID generalisation. Under group shift the interaction is worse:
   driving the student to imitate the teacher's biased soft targets more closely risks
   compressing more spurious correlation into the student's weights.

**Where to go next.** Try varying the correlation probability (90 % -> 70 % -> 50 %) and
observe how the agreement gap changes. At 50 % (random colour) there is no spurious
signal -- Group A and Group B should be equivalent. As correlation increases, the gap
grows. This sweep quantifies how much spurious signal is needed to produce a detectable
agreement drop -- useful for calibrating a production alert threshold.